# QAA Multi-Asset Portfolio Construction

This notebook refactors the original course assignment into a cleaner research workflow.

## Goals

- load and clean the original multi-asset dataset,
- engineer the reduced investment universe,
- study risk-return trade-offs,
- build several portfolio-allocation candidates,
- compare them through historical and simulated metrics.

> Place the original Excel files (`Database_gg3.xlsx` and `Database_gg.xlsx`) inside `data/raw/` before running the notebook end to end.


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing 'src/' and 'notebooks/'.")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import numpy as np
import pandas as pd
from tabulate import tabulate

from qaa.config import PERIODS_PER_YEAR
from qaa.data import load_assignment_data, build_research_universe, compute_log_returns, proxy_equilibrium_weights
from qaa.metrics import (
    annualized_return,
    annualized_volatility,
    annualized_downside_volatility,
    historic_var,
    parametric_var,
    historic_es,
    parametric_es,
    sharpe_ratio,
    sortino_ratio,
    omega_ratio,
    calmar_ratio,
    max_drawdown,
)
from qaa.optimization import (
    portfolio_series,
    long_only_bounds,
    base_constraints,
    group_lower_bound,
    group_upper_bound,
    relative_group_lower_bound,
    solve_min_vol,
    solve_min_downside_vol,
    solve_sortino,
    solve_equal_risk_contribution,
    solve_max_diversification,
    resampled_mean_variance_weights,
    implied_equilibrium_returns,
    build_endogenous_bl_views,
    black_litterman_posterior_mean,
    simulate_block_bootstrap_portfolio_returns,
)
from qaa.backtesting import run_walk_forward_backtest
from qaa.plotting import plot_correlation_heatmap, plot_risk_return_scatter, plot_weights

pd.options.display.float_format = "{:,.4f}".format


## 1. Load the original data

The original script read from hard-coded local Windows paths.  
Here the data-loading logic is centralized in `src/qaa/data.py`.


In [ ]:
data_raw = load_assignment_data()
data_raw.head()

## 2. Baseline asset-universe inspection


In [ ]:
returns_raw = compute_log_returns(data_raw)
rf_annual = returns_raw["US Money Market"].mean() * PERIODS_PER_YEAR
sigma_raw_annual = returns_raw.cov() * PERIODS_PER_YEAR
corr_raw = returns_raw.corr()

asset_vol = returns_raw.std() * np.sqrt(PERIODS_PER_YEAR)
asset_down_vol = annualized_downside_volatility(returns_raw, PERIODS_PER_YEAR)
asset_hist_es = historic_es(returns_raw, PERIODS_PER_YEAR, 0.05)
asset_mu = returns_raw.mean() * PERIODS_PER_YEAR

plot_correlation_heatmap(corr_raw, "Baseline correlation heatmap")
plot_risk_return_scatter(asset_vol, asset_mu, "Risk-return trade-off (baseline universe)", "Volatility", "Annualized return")
plot_risk_return_scatter(asset_down_vol, asset_mu, "Downside-risk trade-off (baseline universe)", "Downside volatility", "Annualized return")
plot_risk_return_scatter(-asset_hist_es, asset_mu, "Expected-shortfall trade-off (baseline universe)", "Historical ES (absolute)", "Annualized return")


## 3. Engineer the reduced research universe

This replicates the original step where the universe is compressed into a cleaner set of investable sleeves:
- blended EM bond,
- blended EU IG bond,
- blended opportunities bucket.


In [ ]:
data = build_research_universe(data_raw)
returns = compute_log_returns(data)

sigma_annual = returns.cov() * PERIODS_PER_YEAR
corr = returns.corr()
mu_annual = returns.mean() * PERIODS_PER_YEAR
down_vol = annualized_downside_volatility(returns, PERIODS_PER_YEAR)
hist_es = historic_es(returns, PERIODS_PER_YEAR, 0.05)
vol_annual = annualized_volatility(returns, PERIODS_PER_YEAR)

plot_correlation_heatmap(corr, "Correlation heatmap (engineered universe)")
plot_risk_return_scatter(vol_annual, mu_annual, "Risk-return trade-off (engineered universe)", "Volatility", "Annualized return")
plot_risk_return_scatter(down_vol, mu_annual, "Downside-risk trade-off (engineered universe)", "Downside volatility", "Annualized return")
plot_risk_return_scatter(-hist_es, mu_annual, "Expected-shortfall trade-off (engineered universe)", "Historical ES (absolute)", "Annualized return")


## 4. Proxy equilibrium returns


In [ ]:
proxy_w = proxy_equilibrium_weights(data)
pi = implied_equilibrium_returns(
    sigma_annual=sigma_annual,
    equilibrium_weights=proxy_w,
    risk_free_rate=rf_annual,
    risk_aversion=4.5,
)

plot_weights(proxy_w, "Proxy equilibrium weights")
plot_risk_return_scatter(vol_annual, pi, "Risk-return trade-off using proxy equilibrium returns", "Volatility", "Implied equilibrium return")
pi.sort_values(ascending=False).to_frame("pi")


## 5. Portfolio constraints

The constrained variants in the original assignment were trying to encode:
- minimum overall European exposure,
- bounded total bond exposure,
- minimum high-yield share inside the bond bucket,
- cap on money-market exposure,
- minimum overall equity exposure,
- minimum EU-equity share inside the equity bucket.


In [ ]:
flag_eu = pd.Series(0, index=returns.columns)
flag_eu[["EU Money Mkt", "EU Bond", "EU Equity", "EU IG Bond"]] = 1

flag_bond = pd.Series(0, index=returns.columns)
flag_bond[["EU Bond", "Global Bond", "EM Bond", "Global Corp. Bond High Yield", "EU IG Bond"]] = 1

flag_hy = pd.Series(0, index=returns.columns)
flag_hy[["Global Corp. Bond High Yield"]] = 1

flag_mm = pd.Series(0, index=returns.columns)
flag_mm[["EU Money Mkt", "US Money Market"]] = 1

flag_equity = pd.Series(0, index=returns.columns)
flag_equity[["EU Equity", "North America Equity", "Pacific Equity", "EM Equity"]] = 1

flag_eu_equity = pd.Series(0, index=returns.columns)
flag_eu_equity[["EU Equity"]] = 1

max_weights_constrained = [0.10, 0.25, 0.30, 0.20, 0.25, 0.25, 0.40, 0.30, 0.30, 0.08, 0.15, 0.35]
bounds_plain = long_only_bounds(returns.columns)
bounds_constrained = long_only_bounds(returns.columns, max_weights=max_weights_constrained)

target_return = 0.10

constraints_plain = base_constraints(returns, target_return=target_return, periods_per_year=PERIODS_PER_YEAR)

constraints_constrained = base_constraints(returns, target_return=target_return, periods_per_year=PERIODS_PER_YEAR) + [
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_eu, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_bond, 0.35)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_bond, 0.25)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_hy, flag_bond, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_mm, 0.12)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_equity, 0.40)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_eu_equity, flag_equity, 0.30)},
]


## 6. Endogenous Black-Litterman views

Instead of hard-coding relative-return views, build them from the latest cross-asset signal snapshot:
- **6M momentum**,
- **current drawdown** over a trailing window,
- **average absolute correlation** as a crude crowding/diversification proxy.

The composite score ranks sleeves endogenously and generates pair views, expected spread returns, and dynamic confidence levels.


In [ ]:
endogenous_views = build_endogenous_bl_views(
    returns,
    periods_per_year=PERIODS_PER_YEAR,
    n_views=3,
    momentum_window=126,
    drawdown_window=126,
    corr_window=63,
    view_scale=0.35,
)

bl_signal_table = endogenous_views.signal_table[
    [
        "momentum_6m",
        "latest_drawdown",
        "avg_abs_corr",
        "vol_annual",
        "composite_score",
        "rank",
    ]
]
bl_view_summary = pd.concat(
    [
        endogenous_views.q.rename("q_annual"),
        endogenous_views.confidences.rename("confidence"),
    ],
    axis=1,
)

display(bl_signal_table)
display(bl_view_summary)
endogenous_views.P


## 7. Solve the optimization problems

This section now broadens the competition from “which single optimized portfolio wins?” to “which **allocator philosophy** is more robust?”  
In addition to the original sleeves, add three challengers:
- **ERC / Equal Risk Contribution**
- **Maximum Diversification**
- **Resampled Mean-Variance** à la Michaud

The Black-Litterman block also uses the endogenous view engine from the previous section.


In [ ]:
def plain_constraints_builder(sample_returns: pd.DataFrame) -> list[dict]:
    return base_constraints(
        sample_returns,
        target_return=target_return,
        periods_per_year=PERIODS_PER_YEAR,
    )

def constrained_constraints_builder(sample_returns: pd.DataFrame) -> list[dict]:
    return base_constraints(
        sample_returns,
        target_return=target_return,
        periods_per_year=PERIODS_PER_YEAR,
    ) + [
        {"type": "ineq", "fun": group_lower_bound, "args": (flag_eu, 0.35)},
        {"type": "ineq", "fun": group_upper_bound, "args": (flag_bond, 0.35)},
        {"type": "ineq", "fun": group_lower_bound, "args": (flag_bond, 0.25)},
        {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_hy, flag_bond, 0.35)},
        {"type": "ineq", "fun": group_upper_bound, "args": (flag_mm, 0.12)},
        {"type": "ineq", "fun": group_lower_bound, "args": (flag_equity, 0.40)},
        {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_eu_equity, flag_equity, 0.30)},
    ]

constraints_plain = plain_constraints_builder(returns)
constraints_constrained = constrained_constraints_builder(returns)
constraints_budget_only = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]

res_markowitz = solve_min_vol(returns, bounds_plain, constraints_plain, PERIODS_PER_YEAR)
res_constrained = solve_min_vol(returns, bounds_constrained, constraints_constrained, PERIODS_PER_YEAR)
res_downside = solve_min_downside_vol(returns, bounds_constrained, constraints_constrained, PERIODS_PER_YEAR)

res_erc = solve_equal_risk_contribution(returns, bounds_plain, constraints_budget_only, PERIODS_PER_YEAR)
res_maxdiv = solve_max_diversification(returns, bounds_plain, constraints_budget_only, PERIODS_PER_YEAR)
resampled_mv = resampled_mean_variance_weights(
    returns,
    bounds_plain,
    plain_constraints_builder,
    periods_per_year=PERIODS_PER_YEAR,
    n_bootstrap=250,
    seed=42,
)

bl_mu = black_litterman_posterior_mean(
    sigma_annual=sigma_annual,
    pi=pi,
    P=endogenous_views.P,
    q=endogenous_views.q,
    confidences=endogenous_views.confidences,
    tau=1 / 15,
)

returns_bl = returns.copy()
returns_bl.loc[:, :] = returns_bl.values - returns_bl.mean().values + (bl_mu.values / PERIODS_PER_YEAR)

res_bl_endogenous = solve_min_vol(
    returns_bl,
    bounds_plain,
    plain_constraints_builder(returns_bl),
    PERIODS_PER_YEAR,
)
res_bl_endogenous_constrained = solve_min_vol(
    returns_bl,
    bounds_constrained,
    constrained_constraints_builder(returns_bl),
    PERIODS_PER_YEAR,
)

constraints_sortino_plain = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
constraints_sortino_constrained = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}] + [
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_eu, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_bond, 0.35)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_bond, 0.25)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_hy, flag_bond, 0.35)},
    {"type": "ineq", "fun": group_upper_bound, "args": (flag_mm, 0.12)},
    {"type": "ineq", "fun": group_lower_bound, "args": (flag_equity, 0.40)},
    {"type": "ineq", "fun": relative_group_lower_bound, "args": (flag_eu_equity, flag_equity, 0.30)},
]

res_sortino = solve_sortino(returns, bounds_plain, constraints_sortino_plain, rf_annual, PERIODS_PER_YEAR)
res_sortino_constrained = solve_sortino(returns, bounds_constrained, constraints_sortino_constrained, rf_annual, PERIODS_PER_YEAR)

allocator_diagnostics = pd.DataFrame(
    {
        "success": [
            res_markowitz.success,
            res_constrained.success,
            res_downside.success,
            res_erc.success,
            res_maxdiv.success,
            True,
            res_bl_endogenous.success,
            res_bl_endogenous_constrained.success,
            res_sortino.success,
            res_sortino_constrained.success,
        ],
        "objective": [
            res_markowitz.fun,
            res_constrained.fun,
            res_downside.fun,
            res_erc.fun,
            res_maxdiv.fun,
            np.nan,
            res_bl_endogenous.fun,
            res_bl_endogenous_constrained.fun,
            res_sortino.fun,
            res_sortino_constrained.fun,
        ],
    },
    index=[
        "Markowitz",
        "Constrained",
        "Downside Constrained",
        "ERC",
        "Max Diversification",
        "Resampled MV",
        "BL Endogenous",
        "BL Endogenous Constrained",
        "Sortino",
        "Sortino Constrained",
    ],
)
allocator_diagnostics.loc["Resampled MV", "bootstrap_success_ratio"] = resampled_mv.success_ratio
allocator_diagnostics


In [ ]:
weights = pd.DataFrame(
    {
        "Markowitz": res_markowitz.x,
        "Constrained": res_constrained.x,
        "Downside Constrained": res_downside.x,
        "ERC": res_erc.x,
        "Max Diversification": res_maxdiv.x,
        "Resampled MV": resampled_mv.weights.values,
        "BL Endogenous": res_bl_endogenous.x,
        "BL Endogenous Constrained": res_bl_endogenous_constrained.x,
        "Sortino": res_sortino.x,
        "Sortino Constrained": res_sortino_constrained.x,
    },
    index=returns.columns,
)

weights.style.format("{:.2%}")


## 8. Inspect portfolio compositions


In [ ]:
for col in weights.columns:
    plot_weights(weights[col], col)

## 9. Historical portfolio return series


In [ ]:
portfolio_returns = pd.DataFrame(
    {name: portfolio_series(returns, weights[name]) for name in weights.columns}
)
portfolio_returns.head()


In [ ]:
ax = (1 + portfolio_returns).cumprod().plot(figsize=(12, 6), grid=True, title="Historical equity curves")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative wealth")


## 10. Forward scenario simulation via multivariate block bootstrap

The original GBM-style simulation was useful pedagogically but weak as a portfolio-risk engine: it imposed thin-tailed Gaussian shocks, constant volatility, static correlations, and independent increments.  
Here the forward scenario layer is replaced by a **multivariate stationary block bootstrap** on the asset-return panel.

Why this is more defensible:

- simulated shocks are sampled jointly across assets, preserving empirical cross-asset dependence;
- blocks preserve some volatility clustering and serial dependence;
- no parametric Gaussian assumption is imposed;
- every portfolio is shocked by the same simulated asset-return scenarios, making comparisons fairer.


In [ ]:
seed = 2800
n_years = 22
n_sims = 1000
avg_block_length = 21  # roughly one trading month

simulated_portfolio_returns = simulate_block_bootstrap_portfolio_returns(
    asset_returns=returns,
    weights=weights,
    n_years=n_years,
    periods_per_year=PERIODS_PER_YEAR,
    n_sims=n_sims,
    avg_block_length=avg_block_length,
    seed=seed,
)

sim_median = pd.DataFrame(
    {name: sims.median(axis=1) for name, sims in simulated_portfolio_returns.items()}
)

ax = (1 + sim_median).cumprod().plot(
    figsize=(12, 6),
    grid=True,
    title="Median forward scenarios: multivariate block bootstrap",
)
ax.set_xlabel("Step")
ax.set_ylabel("Cumulative wealth")

mc_tail_metrics = pd.DataFrame(
    {
        name: {
            "MC VaR 5%": np.percentile(sims.values.flatten(), 5),
            "MC ES 5%": sims.values.flatten()[sims.values.flatten() <= np.percentile(sims.values.flatten(), 5)].mean(),
            "Median terminal wealth": (1 + sims).cumprod().iloc[-1].median(),
            "p5 terminal wealth": (1 + sims).cumprod().iloc[-1].quantile(0.05),
            "p95 terminal wealth": (1 + sims).cumprod().iloc[-1].quantile(0.95),
        }
        for name, sims in simulated_portfolio_returns.items()
    }
).T

mc_tail_metrics


## 11. Rolling out-of-sample walk-forward backtest with transaction costs

This is the credibility upgrade.  
Each allocator is re-estimated only with information available at the rebalance date, then held out-of-sample over the next month. Net returns subtract simple turnover-based transaction costs:

\[
	ext{cost}_t = rac{	ext{bps}}{10{,}000}\sum_i |w_{i,t} - w_{i,t-1}|.
\]

This is intentionally simple but important: it stops the notebook from presenting fully in-sample allocations as if they were tradable backtests.


In [ ]:
estimation_window = 756      # approx. 3 years
rebalance_frequency = 21     # approx. monthly
transaction_cost_bps = 10.0  # one-way portfolio turnover cost proxy


def _budget_only() -> list[dict]:
    return [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]


def _plain_bounds(sample: pd.DataFrame):
    return long_only_bounds(sample.columns)


def _constrained_bounds(sample: pd.DataFrame):
    # Reuse the same max weights, aligned to the engineered universe order.
    return long_only_bounds(sample.columns, max_weights=max_weights_constrained)


def make_walkforward_allocators() -> dict:
    return {
        "Markowitz": lambda x: pd.Series(
            solve_min_vol(
                x,
                _plain_bounds(x),
                base_constraints(x, target_return=target_return, periods_per_year=PERIODS_PER_YEAR),
                PERIODS_PER_YEAR,
            ).x,
            index=x.columns,
        ),
        "Constrained": lambda x: pd.Series(
            solve_min_vol(
                x,
                _constrained_bounds(x),
                constrained_constraints_builder(x),
                PERIODS_PER_YEAR,
            ).x,
            index=x.columns,
        ),
        "Downside Constrained": lambda x: pd.Series(
            solve_min_downside_vol(
                x,
                _constrained_bounds(x),
                constrained_constraints_builder(x),
                PERIODS_PER_YEAR,
            ).x,
            index=x.columns,
        ),
        "ERC": lambda x: pd.Series(
            solve_equal_risk_contribution(x, _plain_bounds(x), _budget_only(), PERIODS_PER_YEAR).x,
            index=x.columns,
        ),
        "Max Diversification": lambda x: pd.Series(
            solve_max_diversification(x, _plain_bounds(x), _budget_only(), PERIODS_PER_YEAR).x,
            index=x.columns,
        ),
        "Resampled MV": lambda x: resampled_mean_variance_weights(
            x,
            _plain_bounds(x),
            plain_constraints_builder,
            periods_per_year=PERIODS_PER_YEAR,
            n_bootstrap=100,
            seed=42,
        ).weights,
        "Sortino Constrained": lambda x: pd.Series(
            solve_sortino(
                x,
                _constrained_bounds(x),
                constraints_sortino_constrained,
                rf_annual,
                PERIODS_PER_YEAR,
            ).x,
            index=x.columns,
        ),
    }


wf = run_walk_forward_backtest(
    returns=returns,
    allocator_builders=make_walkforward_allocators(),
    estimation_window=estimation_window,
    rebalance_frequency=rebalance_frequency,
    transaction_cost_bps=transaction_cost_bps,
)

wf_net = wf.returns_net
wf_gross = wf.returns_gross

ax = (1 + wf_net).cumprod().plot(
    figsize=(12, 6),
    grid=True,
    title="Rolling walk-forward OOS equity curves, net of transaction costs",
)
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative wealth")

wf.diagnostics.groupby("allocator")["success"].mean().rename("optimization_success_rate").to_frame()


In [ ]:
wf_turnover_summary = wf.turnover.agg(["mean", "median", "max"]).T
wf_turnover_summary.columns = ["Mean turnover", "Median turnover", "Max turnover"]

wf_net_metrics = pd.DataFrame(
    {
        "Ann. Return": annualized_return(wf_net, PERIODS_PER_YEAR),
        "Ann. Vol": annualized_volatility(wf_net, PERIODS_PER_YEAR),
        "Downside Vol": annualized_downside_volatility(wf_net, PERIODS_PER_YEAR),
        "Sharpe": sharpe_ratio(wf_net, rf_annual, PERIODS_PER_YEAR),
        "Sortino": sortino_ratio(wf_net, rf_annual, PERIODS_PER_YEAR),
        "Hist VaR 5%": historic_var(wf_net, PERIODS_PER_YEAR, 0.05),
        "Hist ES 5%": historic_es(wf_net, PERIODS_PER_YEAR, 0.05),
        "MaxDD": max_drawdown(wf_net, 3000),
        "Calmar": calmar_ratio(wf_net, 252, PERIODS_PER_YEAR),
    }
).sort_values("Sharpe", ascending=False)

pd.concat(
    [wf_net_metrics, wf_turnover_summary.reindex(wf_net_metrics.index)],
    axis=1,
)


## 12. Static in-sample risk metrics


In [ ]:
risk_metrics = pd.DataFrame(
    {
        "Downside Volatility": annualized_downside_volatility(portfolio_returns, PERIODS_PER_YEAR),
        "Historical VaR (5%)": historic_var(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Historical ES (5%)": historic_es(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Parametric VaR (5%)": parametric_var(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Parametric ES (5%)": parametric_es(portfolio_returns, PERIODS_PER_YEAR, 0.05),
        "Max Drawdown": max_drawdown(portfolio_returns, 3000),
    }
).T

risk_metrics


## 13. Static in-sample risk-adjusted performance metrics


In [ ]:
risk_adjusted = pd.DataFrame(
    {
        "Sharpe": sharpe_ratio(portfolio_returns, rf_annual, PERIODS_PER_YEAR),
        "Sortino": sortino_ratio(portfolio_returns, rf_annual, PERIODS_PER_YEAR),
        "Omega (10%)": omega_ratio(portfolio_returns, 0.10, PERIODS_PER_YEAR),
        "Calmar": calmar_ratio(portfolio_returns, 252, PERIODS_PER_YEAR),
    }
).T

risk_adjusted


## 14. Final remarks

This notebook now goes one step further than the original course assignment:

- reusable code in `src/qaa`,
- reproducible relative paths,
- explicit notebook narrative,
- a broader allocator horse-race (ERC, MaxDiv, resampled MV),
- an endogenous Black-Litterman layer where views are generated from cross-asset signals rather than fixed by hand.

The notebook now includes a rolling OOS walk-forward layer, turnover-based transaction costs, and multivariate block-bootstrap forward scenarios. Natural next enhancements are richer cost models, regime-conditioned scenario generation, and validation against investable ETF proxies.
